In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# dataset path
dataset_path = "D:\prash projects\skin desease pridictor\Raw_Dataset"   # apna folder name dalna

img_size = 224
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 2828 images belonging to 7 classes.
Found 704 images belonging to 7 classes.


In [2]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_data.num_classes, activation='softmax')
])

In [3]:
print(train_data.class_indices)

{'ch_Chickenpox_Varicela': 0, 'hz_Herpes': 1, 'lp_Lupus': 2, 'me_Melanoma': 3, 'mk_Monkeypox': 4, 'ms_Measles_Sarampion': 5, 'sc_Scabies_sarna': 6}


In [4]:
class_names = list(train_data.class_indices.keys())
print(class_names)

['ch_Chickenpox_Varicela', 'hz_Herpes', 'lp_Lupus', 'me_Melanoma', 'mk_Monkeypox', 'ms_Measles_Sarampion', 'sc_Scabies_sarna']


In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(7, activation='softmax')  # 7 classes
])

In [6]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [24]:
from PIL import Image
import os

dataset_path = r"D:\prash projects\skin desease pridictor\Raw_Dataset"# apna folder

bad_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            img = Image.open(file_path)
            img.verify()  # check image
        except:
            bad_files.append(file_path)

# delete bad files
for file in bad_files:
    os.remove(file)

print("Removed files:", len(bad_files))

Removed files: 0


In [25]:
import os
print(os.path.exists(dataset_path))

True


In [26]:
print(os.listdir(dataset_path))

['ch_Chickenpox_Varicela', 'hz_Herpes', 'lp_Lupus', 'me_Melanoma', 'mk_Monkeypox', 'ms_Measles_Sarampion', 'sc_Scabies_sarna']


In [27]:
train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='sparse',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='sparse',
    subset='validation'
)

Found 2672 images belonging to 7 classes.
Found 666 images belonging to 7 classes.


In [28]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 39s 466ms/step - accuracy: 0.4858 - loss: 2.0718 - val_accuracy: 0.5991 - val_loss: 1.0505
Epoch 2/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 35s 416ms/step - accuracy: 0.5734 - loss: 1.0687 - val_accuracy: 0.6276 - val_loss: 0.9545
Epoch 3/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 34s 404ms/step - accuracy: 0.6067 - loss: 0.9849 - val_accuracy: 0.7012 - val_loss: 0.8791
Epoch 4/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 32s 376ms/step - accuracy: 0.6216 - loss: 0.9430 - val_accuracy: 0.7132 - val_loss: 0.8365
Epoch 5/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 31s 371ms/step - accuracy: 0.6445 - loss: 0.8945 - val_accuracy: 0.7252 - val_loss: 0.7961


In [29]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [30]:
base_model.trainable = True

for layer in base_model.layers[:-20]:  # last 20 layers train honge
    layer.trainable = False

In [31]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [32]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 41s 437ms/step - accuracy: 0.6160 - loss: 1.0000 - val_accuracy: 0.7312 - val_loss: 0.7349
Epoch 2/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 35s 417ms/step - accuracy: 0.6793 - loss: 0.8180 - val_accuracy: 0.7523 - val_loss: 0.6898
Epoch 3/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 34s 410ms/step - accuracy: 0.7324 - loss: 0.6919 - val_accuracy: 0.7568 - val_loss: 0.7019
Epoch 4/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 35s 422ms/step - accuracy: 0.7657 - loss: 0.6051 - val_accuracy: 0.7583 - val_loss: 0.6962
Epoch 5/5
84/84 ━━━━━━━━━━━━━━━━━━━━ 35s 419ms/step - accuracy: 0.7975 - loss: 0.5111 - val_accuracy: 0.7492 - val_loss: 0.7002


In [34]:
restore_best_weights=True

In [35]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'best_skin_model.h5', 'Raw_Dataset', 'Untitled.ipynb']


In [37]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

# model load
model = tf.keras.models.load_model("best_skin_model.h5")

# classes
class_names = [
 'ch_Chickenpox_Varicela',
 'hz_Herpes',
 'lp_Lupus',
 'me_Melanoma',
 'mk_Monkeypox',
 'ms_Measles_Sarampion',
 'sc_Scabies_sarna'
]

# test image
img_path = "D:/prash projects/skin desease pridictor/test.jpg"   # apni image ka path

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

print("Prediction:", class_names[np.argmax(prediction)])
print("Confidence:", np.max(prediction))

FileNotFoundError: [Errno 2] No such file or directory: 'D:/prash projects/skin desease pridictor/test.jpg'

In [38]:
import os

path = "D:/prash projects/skin desease pridictor"
print(os.listdir(path))

['.ipynb_checkpoints', 'best_skin_model.h5', 'Raw_Dataset', 'Untitled.ipynb']


In [39]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'best_skin_model.h5', 'Raw_Dataset', 'test.jpg.webp', 'Untitled.ipynb']


In [40]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

# model load
model = tf.keras.models.load_model("best_skin_model.h5")

# class names
class_names = [
 'ch_Chickenpox_Varicela',
 'hz_Herpes',
 'lp_Lupus',
 'me_Melanoma',
 'mk_Monkeypox',
 'ms_Measles_Sarampion',
 'sc_Scabies_sarna'
]

# image path (same folder me hai to simple likho)
img_path = "test.jpg"

# preprocess
img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

# predict
prediction = model.predict(img_array)

print("Prediction:", class_names[np.argmax(prediction)])
print("Confidence:", np.max(prediction))

FileNotFoundError: [Errno 2] No such file or directory: 'test.jpg'

In [41]:
import os
print(os.getcwd())

D:\prash projects\skin desease pridictor


In [42]:
print(os.listdir())

['.ipynb_checkpoints', 'best_skin_model.h5', 'Raw_Dataset', 'test.jpg.webp', 'Untitled.ipynb']


In [43]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'best_skin_model.h5', 'Raw_Dataset', 'test.jpg.webp', 'Untitled.ipynb']


In [44]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'best_skin_model.h5', 'Raw_Dataset', 'test.jpg', 'Untitled.ipynb']


In [45]:
img_path = "test.jpg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

print("Prediction:", class_names[np.argmax(prediction)])
print("Confidence:", round(np.max(prediction)*100,2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 666ms/step
Prediction: sc_Scabies_sarna
Confidence: 50.74 %


In [46]:
import numpy as np

probs = prediction[0]
top3_idx = probs.argsort()[-3:][::-1]

for i in top3_idx:
    print(class_names[i], ":", round(probs[i]*100,2), "%")

sc_Scabies_sarna : 50.74 %
ms_Measles_Sarampion : 21.89 %
hz_Herpes : 8.61 %


In [47]:
|model.save("skin_model.keras")

SyntaxError: invalid syntax (3409461946.py, line 1)

In [48]:
model.save("skin_model.keras")

In [50]:
model.save("skin_model.keras")

In [51]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'best_skin_model.h5', 'Raw_Dataset', 'skin_model.keras', 'test.jpg', 'Untitled.ipynb']


In [52]:
import os
print(os.getcwd())
print(os.listdir())

D:\prash projects\skin desease pridictor
['.ipynb_checkpoints', 'best_skin_model.h5', 'skin_model.keras', 'test.jpg', 'Untitled.ipynb']


In [53]:
from tensorflow.keras.models import load_model

model = load_model("skin_model.keras")
print("Model loaded successfully")

Model loaded successfully


C:\Users\prash\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 52 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [54]:
from tensorflow.keras.models import load_model

model = load_model("skin_model.keras", compile=False)
print("Model loaded successfully")

Model loaded successfully


In [55]:
import os

img_path = r"D:\prash projects\skin desease pridictor\test.jpg"
print(os.path.exists(img_path))

True
